# WS 6.2: Variation

In this worksheet, we explore different ways to measure how *spread out* data is. We will work with temperature data from two US cities and build up to the **standard deviation** step by step.

In [ ]:
#@title Run this cell to set up plotting tools
import numpy as np
import matplotlib.pyplot as plt

def stacked_dotplot(*groups, labels=None, colors=None, xlabel='', title=''):
    """Create a histodot-style stacked dot plot."""
    if colors is None:
        colors = ['steelblue', 'coral', 'seagreen', 'mediumpurple']

    groups = [np.asarray(g, dtype=float) for g in groups]
    n = len(groups)
    all_vals = np.concatenate(groups)
    data_range = all_vals.max() - all_vals.min()
    bin_width = data_range / 20 if data_range > 0 else 1
    dot_radius = bin_width * 0.45  # slightly smaller than half-bin for gap

    # Snap each value to its bin center
    def to_bin(v):
        return np.round(v / bin_width) * bin_width

    # First pass: bin values and compute stacking
    max_stack = 0
    group_data = []
    for values in groups:
        binned = to_bin(values)
        order = np.argsort(binned)
        binned_sorted = binned[order]

        stack = np.zeros(len(binned_sorted))
        for i in range(1, len(binned_sorted)):
            if np.isclose(binned_sorted[i], binned_sorted[i - 1]):
                stack[i] = stack[i - 1] + 1

        group_data.append((binned_sorted, stack))
        max_stack = max(max_stack, int(stack.max()))

    group_spacing = (max_stack + 2.5) * bin_width

    # Second pass: draw circles in data coordinates
    fig, ax = plt.subplots(figsize=(10, 3))

    for g, (binned_sorted, stack) in enumerate(group_data):
        y_base = g * group_spacing if n > 1 else 0
        color = colors[g % len(colors)]

        for x, s in zip(binned_sorted, stack):
            cy = y_base + dot_radius + s * bin_width
            circle = plt.Circle((x, cy), dot_radius,
                                fc=color, ec='black', lw=0.8, zorder=3)
            ax.add_patch(circle)

        if labels and g < len(labels):
            ax.text(all_vals.min() - bin_width * 1.5, y_base + dot_radius,
                    labels[g], ha='right', va='center',
                    fontsize=13, fontweight='bold')

    ax.set_aspect('equal')
    ax.autoscale()
    x_pad = bin_width
    ax.set_xlim(all_vals.min() - x_pad * 3, all_vals.max() + x_pad)
    ax.set_yticks([])
    ax.set_xlabel(xlabel, fontsize=12)
    if title:
        ax.set_title(title, fontsize=14)
    ax.grid(axis='x', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    plt.tight_layout()
    plt.show()

## Part 1: How can we measure variability?

Consider the temperatures on April 14 for 11 different years, for two cities in the United States: Des Moines, Iowa (DM) and San Francisco, CA (SF). (Data adapted from [the StatKey website](https://www.lock5stat.com/StatKey/descriptive_1_quant_1_cat/descriptive_1_quant_1_cat.html))

We can store data in Python using NumPy arrays, as we learned in WS 6.1:

In [ ]:
dm = np.array([2.9, 5, 6.9, 11.6, 11.8, 12.4, 12.6, 13.3, 15.7, 15.9, 21.4])
sf = np.array([2.8, 11.2, 11.9, 12, 12.3, 13.2, 13.4, 14, 14.1, 14.7, 21.7])

We have set up a plotting function called `stacked_dotplot()` that creates dot plots from data. Here is a dot plot of the Des Moines temperatures:

In [ ]:
stacked_dotplot(dm, xlabel='Temperature (°C)', title='Des Moines April 14 Temperatures')

NumPy has a handy function `.mean()` that calculates the mean of an array. For example:

In [ ]:
print("The DM mean is", dm.mean())

**Exercise 1.1:** Use `.mean()` to calculate the mean of the San Francisco temperatures.

In [ ]:
# Your code here

**Exercise 1.2:** Use `stacked_dotplot()` to create a dot plot of the San Francisco temperatures. Use the Des Moines plot above as a guide -- you just need to change the data and the title.

In [ ]:
# Your code here

**Exercise 1.3:** Looking at your two dot plots, which city's temperatures appear more spread out? How can you tell?

*Your answer:*

## Part 2: Understanding deviations

A **deviation** measures how far each data value is from the mean. We calculate it by subtracting the mean from each data value:

$$\text{deviation} = \text{value} - \text{mean}$$

For example, if a temperature is 15.7 and the mean is 11.8, the deviation is $15.7 - 11.8 = 3.9$.

In Python, if we subtract a single number from a NumPy array, it subtracts from *every* element. For example:

In [ ]:
example = np.array([10, 20, 30])
example - 5

**Exercise 2.1:** Using this idea, calculate the deviations for Des Moines. Store the result in a variable called `dm_deviations`, then display it.

*Hint:* Subtract `dm.mean()` from `dm`.

In [ ]:
# Your code here

**Exercise 2.2:** Now plot the DM deviations using `stacked_dotplot()`. Use `xlabel='Deviation from mean (°C)'` and an appropriate title.

In [ ]:
# Your code here

**Exercise 2.3:** Compare your plot of the deviations with the original DM temperature plot from Part 1. What is similar and what is different?

*Your answer:*

One thing that seems like it could help is to take the "average of the deviations" -- taking the mean of the set of 11 deviations. The "average of the deviations" could give a sense of how much, on average, data values differ from the overall mean.

**Exercise 2.4:** Calculate the "average of the deviations" by taking the mean of `dm_deviations`.

In [ ]:
# Your code here

**Exercise 2.5:** Find a member of an ODD group and compare your "average of the deviations" with the "average of the deviations" of the other city. What do you notice? Is the "average of the deviations" a useful comparison?

*Your answer:*

## Part 3: The Mean Absolute Deviation (MAD)

The problem with the "average of the deviations" is that positive and negative deviations cancel out! One way to fix this is to take the **absolute value** of each deviation, which makes every number positive while keeping the same distance from 0.

In Python, the `abs()` function computes the absolute value. For example:

In [ ]:
abs(np.array([-3, -1, 0, 2, 5]))

**Exercise 3.1:** Calculate the absolute deviations for Des Moines. Store the result in a variable called `abs_dm_deviations`, then display the values.

In [ ]:
# Your code here

**Exercise 3.2:** Plot the absolute deviations using `stacked_dotplot()`. Use `xlabel='Absolute deviation (°C)'` and an appropriate title.

In [ ]:
# Your code here

**Exercise 3.3:** Compare your plot of the absolute deviations with the deviations plot from Part 2. What is similar and what is different?

*Your answer:*

**Exercise 3.4:** Calculate the **mean absolute deviation** by taking the mean of `abs_dm_deviations`.

In [ ]:
# Your code here

**Exercise 3.5:** What do you think the mean absolute deviation tells you about the data set?

*Your answer:*

**Exercise 3.6:** Find a member of an ODD group and compare your mean absolute deviation with the mean absolute deviation of the other city. What do you notice?

*Your answer:*

## Part 4: Variance and the standard deviation

Another way to make all the deviations positive is to **square** them. Squaring turns out to have some advantages mathematically (for example, it is easier to take a derivative of a squared function).

In Python, we can square values using the exponent operator `**`. For example:

In [ ]:
np.array([-3, -1, 0, 2, 5]) ** 2

**Exercise 4.1:** Calculate the squared deviations for Des Moines. Store the result in a variable called `squared_dm_deviations`, then display it.

In [ ]:
# Your code here

**Exercise 4.2:** Plot the squared deviations using `stacked_dotplot()`. Use `xlabel='Squared deviation (°C²)'` and an appropriate title.

In [ ]:
# Your code here

**Exercise 4.3:** Compare your plot of the squared deviations with the absolute deviations plot from Part 3. What is similar and what is different?

*Your answer:*

**Exercise 4.4:** Calculate the *mean squared deviation* by calling `.mean()` on `squared_dm_deviations`.

In [ ]:
# Your code here

**Exercise 4.5:** How would you interpret the mean squared deviation?

*Your answer:*

The mean squared deviation is defined as the **variance** of the data (with a technical adjustment we will discuss later). However, note that the variance is in a squared unit -- it would be measured in degrees$^2$ Celsius, whereas the original data was measured in degrees Celsius.

To adjust for this, we can calculate the *root mean squared deviation*, i.e. the square root of the mean squared deviation. This is (approximately) the **standard deviation**.

NumPy already has a function `np.sqrt()` that we can use on a NumPy array. For example:


In [ ]:
np.sqrt(np.array([9, 4, 16, 81, 49]))


**Exercise 4.6:** Calculate the root mean squared deviation, using the value you got in Exercise 4.4 above.

In [ ]:
# your code here

## Part 5: Comparing the two cities

Now let's put it all together and compare Des Moines and San Francisco.

The `stacked_dotplot()` function can plot two groups at once. Just pass both arrays, and use the `labels` option to label them:

In [ ]:
stacked_dotplot(dm, sf, labels=['DM', 'SF'],
                xlabel='Temperature (°C)',
                title='April 14 Temperatures')

**Exercise 5.1:** Looking at this combined plot, which city has more variability? How can you tell visually?

*Your answer:*

**Exercise 5.2:** Find a member of an ODD group and compare your standard deviation (from Exercise 4.6) with the standard deviation of the other city. What do you notice? Is comparing the **standard deviation** between the two cities similar to comparing the **mean absolute deviation** between the two cities?

*Your answer:*

**Exercise 5.3:** In your own words, explain what the standard deviation tells you about a data set.

*Your answer:*

## Part 6: The `.std()` function

It turns out that NumPy has a built-in function `.std()` that calculates the standard deviation directly, without needing to do all the steps by hand! We will use this function frequently in future activities.

For example:

In [ ]:
print("SF standard deviation:", sf.std())

**Exercise 6.1:** Use `.std()` to calculate the standard deviation of the DM temperatures. Does it match the value you calculated by hand in Exercise 4.6?

In [ ]:
# Your code here

## Part 7: How Reliable Is a Model?

You have already explored the Climate Change model in NetLogo by hand -- adjusting sliders, adding CO2, and watching the temperature respond. But here is a question: if you run the exact same model with the exact same settings twice, do you get the exact same result?

In this part, you will use a tool called **BehaviorSpace** to run the Climate Change model 20 times automatically, with identical settings each time. Then you will use what you have learned about mean and standard deviation to analyze the results.

**Exercise 7.1:** You are going to run the Climate Change model 20 times with exactly the same settings. Do you think the final temperature will be the same each time, or different? Why?

*Your answer:*

### Setting up the experiment in BehaviorSpace

BehaviorSpace is a tool built into NetLogo that runs your model many times automatically and records the results. Follow the steps below carefully.

**Step 1.** Open NetLogo 7. Go to **File > Models Library > Sample Models > Earth Science > Climate Change**. Click **Open**.

**Step 2.** Go to **Tools > BehaviorSpace**. Click **New** to create a new experiment.

**Step 3.** Fill in the experiment dialog with the following settings:

**Experiment name:** `temperature-variability`
This is just a label -- you can name it anything.

**Repetitions:** `20`
This tells BehaviorSpace how many times to run the model.

**Metrics:** `temperature`
This is the variable that BehaviorSpace will record at the end of each run.

**UNCHECK this box:** `Run metrics every step`.  We only want to see the final temperature at the end of our simulation, not the temperature at every tick.

**Setup commands:**
```
setup
repeat 8 [ add-CO2 ]
```
These commands run before each experiment begins. `setup` initializes the model, and `repeat 8 [ add-CO2 ]` adds CO2 to the atmosphere 8 times.

**Go commands:** `go`
This is the command that advances the model one tick. BehaviorSpace will call it repeatedly until the time limit.

**Stop condition:** *(leave blank)*
We use the time limit instead.

**Time limit:** `500`
Each run will stop after 500 ticks.

**Step 4.** Click "OK". Now you have an experiment. Select the experiment and click "Run".

**Step 5.** Click **OK**. Then click **Run**.

In the dialog that comes up, find the checkboxes at the bottom. Make sure the following are **unchecked**:
- Update view
- Update plots and monitors
Unchecking these makes the experiment run much faster.

**Step 6.** We want the **Table** format, which is easiest to read into Python.  Next to "Table output", click "Browse..." and choose a location on your computer that you can find easily (like "Downloads", or wherever works for you). Click OK and wait for all 20 runs to finish (it should take just a couple seconds).

### Analyzing your results

Now bring your BehaviorSpace results into Python. Save your CSV file to Google Drive (in your QR folder), just like you did with the NASA temperature file in WS 6.1.

First, mount your Google Drive if you haven't already:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Exercise 7.2:** Import your BehaviorSpace CSV using `pd.read_csv()` with `skiprows=6`. The table format has 6 header rows before the data starts, so we need to skip them -- just like you skipped the extra row in WS 6.1. Then extract the `"temperature"` column.

In [ ]:
# Your code here

**Exercise 7.3:** Use `stacked_dotplot()` to plot your 20 temperature values.

In [ ]:
# Your code here

**Exercise 7.4:** Calculate the mean and standard deviation of your 20 temperature values using `.mean()` and `.std()`.

In [ ]:
# Your code here

**Exercise 7.5:** Look at your dot plot and your mean/standard deviation. Are all 20 values identical? How spread out are they? Describe what you see.

*Your answer:*

**Exercise 7.6:** Find a member of an ODD group. They ran the same model but with a different amount of CO2. Share your mean and standard deviation with each other.

- Your condition: **8x CO2** -- Your mean: \_\_\_\_\_\_\_\_ -- Your SD: \_\_\_\_\_\_\_\_
- Their condition: **4x CO2** -- Their mean: \_\_\_\_\_\_\_\_ -- Their SD: \_\_\_\_\_\_\_\_

Which condition produced higher temperatures? Which had more variability? What might explain the differences?

*Your answer:*

**Exercise 7.7:** Imagine a scientist ran this model only once and reported that single temperature as their result. Based on what you found today, what is the problem with that approach?

*Your answer:*

-----
Created by Ethan C. Brown, assisted by Claude Code.  Inspired by the "Comparing Hand Spans" activity from EPSY 3264 at the University of Minnesota.